# Shrike AI Lab - Training Ops Dashboard

This notebook gives repeatable commands for diagnostics, training status, and intervention decisions on Windows.

In [ ]:
from pathlib import Path
import subprocess
import shlex

repo = Path(r"D:/LocalProjects/shrike-ai-lab")
python_exe = repo / '.venv/Scripts/python.exe'
powershell_exe = 'powershell'

print('Repo:', repo)
print('Python:', python_exe)
print('Python exists:', python_exe.exists())

def run_command(args, timeout=120, tail_lines=80):
    """Run external command safely in notebook cells.

    - Captures output (prevents notebook UI overload)
    - Enforces timeout
    - Kills process tree on timeout (Windows taskkill /T /F)
    """
    cmd_preview = ' '.join(shlex.quote(str(a)) for a in args)
    print(f'Running: {cmd_preview}')
    proc = subprocess.Popen(
        [str(a) for a in args],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    timed_out = False
    output = ''
    try:
        output, _ = proc.communicate(timeout=timeout)
    except subprocess.TimeoutExpired:
        timed_out = True
        subprocess.run(
            ['taskkill', '/PID', str(proc.pid), '/T', '/F'],
            capture_output=True,
            text=True,
            check=False,
        )
        output, _ = proc.communicate()

    lines = output.splitlines()
    if lines:
        print('\n'.join(lines[-tail_lines:]))
    else:
        print('(no output)')

    rc = proc.returncode if proc.returncode is not None else -9
    print(f'Exit code: {rc}')
    print(f'Timed out: {timed_out}')
    return rc, timed_out, output

In [ ]:
# Full status + diagnostics + summaries
rc, timed_out, _ = run_command([
    powershell_exe, '-NoProfile', '-ExecutionPolicy', 'Bypass',
    '-File', str(repo / 'scripts/ops-status.ps1')
] , timeout=180)

In [ ]:
# Intervention board only (with JSON report)
rc, timed_out, _ = run_command([
    powershell_exe, '-NoProfile', '-ExecutionPolicy', 'Bypass',
    '-File', str(repo / 'scripts/ops-results.ps1'),
    '-Hours', '24',
    '-MinSuccessRuns', '2',
    '-MaxFailRate', '0.5'
] , timeout=120)

In [ ]:
# Example A/B gate run (edit model names before running)
rc, timed_out, _ = run_command([
    powershell_exe, '-NoProfile', '-ExecutionPolicy', 'Bypass',
    '-File', str(repo / 'scripts/ops-ab-gate.ps1'),
    '-Project', 'shared',
    '-Task', 'code_review',
    '-BaselineModel', 'phi3-local',
    '-CandidateModel', 'phi3-local',
    '-Limit', '20'
] , timeout=300)

## Tip
Use Run All to execute your recurring checks in order.

If a command fails, verify the venv path and that PowerShell execution policy is bypassed for this session.

In [ ]:
# One-shot desktop notification if READY_REVIEW queue is non-empty
rc, timed_out, _ = run_command([
    powershell_exe, '-NoProfile', '-ExecutionPolicy', 'Bypass',
    '-File', str(repo / 'scripts/ops-notify-readyreview.ps1')
] , timeout=180)